In [1]:
workdir="05-automatic_curations"
draft_models="$workdir/00-input_models"

mkdir -p $workdir $draft_models

### Adjusting the source code of MEMOTE

Since scraping the `html` report output of MEMOTE turned out not to be easily parsable, I searched to expose or to implement another formatting of the MEMOTE test report file. Apparently, MEMOTE silently supports `json` output formatting, but does not expose this method to the CLI tool. I slightly adjusted the API and the CLI source code to expose this method and to clear a hard-coded preference for `html` outputs.

The implemented changes are the following:

`memote/suite/cli/report.py`

- Added a `click` flag for json report generation (ll. 64-68)
- Added `json` as config variable to the `snapshot` function (ll. 136)
- Added a case for a positive `json` flag, including adjusting the report filename and the use of `json` report generation method.

Generate a `json` memote report for all assemblies.

In [5]:
mkdir -p $workdir/memote
dir -1 $draft_models | xargs basename -s .xml | xargs -t -P 3 -I % bash -c "
echo 'Creating report for %'
memote report snapshot --json --filename $draft_models/%.html $draft_models/%.xml &> /dev/null
mv $draft_models/%.html $workdir/memote/
mv $draft_models/%.json $workdir/memote/"
2>/dev/null

bash -c ''$'\n''echo '\''Creating report for C_beijerinckii_13821'\'''$'\n''memote report snapshot --json --filename 05-automatic_curations/00-input_models/C_beijerinckii_13821.html 05-automatic_curations/00-input_models/C_beijerinckii_13821.xml &> /dev/null'$'\n''mv 05-automatic_curations/00-input_models/C_beijerinckii_13821.html 05-automatic_curations/memote/'$'\n''mv 05-automatic_curations/00-input_models/C_beijerinckii_13821.json 05-automatic_curations/memote/'
bash -c ''$'\n''echo '\''Creating report for C_beijerinckii_791'\'''$'\n''memote report snapshot --json --filename 05-automatic_curations/00-input_models/C_beijerinckii_791.html 05-automatic_curations/00-input_models/C_beijerinckii_791.xml &> /dev/null'$'\n''mv 05-automatic_curations/00-input_models/C_beijerinckii_791.html 05-automatic_curations/memote/'$'\n''mv 05-automatic_curations/00-input_models/C_beijerinckii_791.json 05-automatic_curations/memote/'
Creating report for C_beijerinckii_13821
Creating report for C_beijeri

### Curating the models

Parse them all and output summarising tables to go through manually using the `Python` script `make_curation_db.py`.

In [5]:
mkdir -p $workdir/01-duplicates_imbalances
python utils/make_curation_db.py $workdir/memote $draft_models $workdir/01-duplicates_imbalances

'' is not a valid SBML 'SId'.
Set parameter Username
Academic license - for non-commercial use only - expires 2026-09-26
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.
'' is not a valid SBML 'SId'.


Screen the imbalances and the duplicate reactions manually using SEED reference entries and checking the presence of similar reactions in the model using Fluxer webserver.

We have provided an example database with decisions about duplicate reactions we encountered in Excel file `curations.xlsx`. In the sheet `duplicate_reactions`, a colour code has been applied while going through the reactions manually and converted into an additional `Decision` column by which the chosen action is identified. In addition, the sheets `corrected_balances` and `ignored_imbalances` are self-explanatory.

Add new cases or adjust existing ones as you like.

Implement all decisions in this Excel file in all models using `Python` script `curate_models.py`. This script does the following:

1. Change or remove imbalanced reactions
2. Select which reactions of a duplicate reaction pair should be retained according to the earlier made `Decision` column.
3. Write the curated model away in SBML format

In [12]:
mkdir -p $workdir/01-duplicates_imbalances/logs $workdir/01-duplicates_imbalances/models
dir -1 $draft_models | xargs basename -s .xml | xargs -t -P 12 -I % bash -c "
python utils/curate_models.py $draft_models/%.xml $workdir/01-duplicates_imbalances/models/%.xml curation_DB.xlsx \
> $workdir/01-duplicates_imbalances/logs/%.log 2> $workdir/01-duplicates_imbalances/logs/%.err"

bash -c ''$'\n''python utils/curate_models.py 05-automatic_curations/00-input_models/C_beijerinckii_13821.xml 05-automatic_curations/01-duplicates_imbalances/C_beijerinckii_13821.xml curation_DB.xlsx > 05-automatic_curations/01-duplicates_imbalances/logs/C_beijerinckii_13821.log 2> 05-automatic_curations/01-duplicates_imbalances/logs/C_beijerinckii_13821.err'
bash -c ''$'\n''python utils/curate_models.py 05-automatic_curations/00-input_models/C_beijerinckii_791.xml 05-automatic_curations/01-duplicates_imbalances/C_beijerinckii_791.xml curation_DB.xlsx > 05-automatic_curations/01-duplicates_imbalances/logs/C_beijerinckii_791.log 2> 05-automatic_curations/01-duplicates_imbalances/logs/C_beijerinckii_791.err'


### Completing annotations

#### SBO terms

This is something we can do with the `SBOannotator` tool of the Dräger lab. I've adjusted the source code a bit so it can be more easily wrapped into a for loop. This includes calling system `argv` values and setting some defaults for the `SBOannotator` class. Both scripts have been moved out of their package repository into this local folder for convenience.

Moreover, to enable concurrent use of the SBO annotations database, I've added a boolean flag in the SBOannotator class, that leaves an existing `create_dbs` file untouched. Since all `SBOannotator` processes are instructed to create the same DB file, it makes sense to make them use one created by an earlier `SBOannotator` instance.

In [20]:
mkdir -p $workdir/02-SBO/models $workdir/02-SBO/logs
dir -1 $workdir/01-duplicates_imbalances/models | xargs basename -s .xml | xargs -t -I % bash -c "
python utils/SBOannotator_wrapper.py $workdir/01-duplicates_imbalances/models/%.xml \
$workdir/02-SBO/models/%.xml \
utils/create_dbs \
> $workdir/02-SBO/logs/%.log 2>$workdir/02-SBO/logs/%.err"

bash -c ''$'\n''python utils/SBOannotator_wrapper.py 05-automatic_curations/01-duplicates_imbalances/models/C_beijerinckii_13821.xml 05-automatic_curations/02-SBO/models/C_beijerinckii_13821.xml utils/create_dbs > 05-automatic_curations/02-SBO/logs/C_beijerinckii_13821.log 2>05-automatic_curations/02-SBO/logs/C_beijerinckii_13821.err'
bash -c ''$'\n''python utils/SBOannotator_wrapper.py 05-automatic_curations/01-duplicates_imbalances/models/C_beijerinckii_791.xml 05-automatic_curations/02-SBO/models/C_beijerinckii_791.xml utils/create_dbs > 05-automatic_curations/02-SBO/logs/C_beijerinckii_791.log 2>05-automatic_curations/02-SBO/logs/C_beijerinckii_791.err'


#### Other annotation terms

The other annotation fields are taken care of by `ModelPolisher`, a tool by the Palsson lab. The `Docker` container is currently being updated by the Dräger lab, so I stick to the original Palsson version by calling it from its local compose configuration file.

In [2]:
mkdir -p $workdir/03-annotations/models $workdir/03-annotations/logs

In [3]:
dir -1 $workdir/01-duplicates_imbalances/models | xargs -t -I % bash -c "
cd $HOME/bin/ModelPolisher
docker compose -f docker-compose.yml run -T --rm \
-v $(pwd)/$workdir/01-duplicates_imbalances/models:/models/ \
-v $(pwd)/$workdir/03-annotations/models:/output/ \
-v $(pwd)/$workdir/03-annotations/logs:/logs/ \
polisher java -jar /ModelPolisher-2.1-beta.jar --input=/models/% --output=/output/% \
--annotate-with-bigg=true --check-mass-balance=true --log-file=/logs/%.log"

bash -c ''$'\n''cd /home/lucas/bin/ModelPolisher'$'\n''docker compose -f docker-compose.yml run -T --rm -v /mnt/DATA/PhD/WPs/WP1/clostridia_models/05-automatic_curations/01-duplicates_imbalances/models:/models/ -v /mnt/DATA/PhD/WPs/WP1/clostridia_models/05-automatic_curations/03-annotations/models:/output/ -v /mnt/DATA/PhD/WPs/WP1/clostridia_models/05-automatic_curations/03-annotations/logs:/logs/ polisher java -jar /ModelPolisher-2.1-beta.jar --input=/models/C_beijerinckii_13821.xml --output=/output/C_beijerinckii_13821.xml --annotate-with-bigg=true --check-mass-balance=true --log-file=/logs/C_beijerinckii_13821.xml.log'
WARN[0000] /home/lucas/bin/ModelPolisher/docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion 
[+] Running 0/2
 ⠋ Container modelpolisher_adb     Starti...                               0.1s 
 ⠋ Container modelpolisher_biggdb  Sta...                                  0.1s 
[+] Running 0/2
 ⠙ Containe

### Final MEMOTE

In [4]:
mkdir -p $workdir/memote_final
dir -1 $workdir/03-annotations/models | xargs basename -s .xml | xargs -t -P 3 -I % bash -c "
echo 'Creating report for %'
memote report snapshot --json --filename $workdir/memote_final/%.html \
$workdir/03-annotations/models/%.xml &> /dev/null"
2>/dev/null

bash -c ''$'\n''echo '\''Creating report for C_beijerinckii_13821'\'''$'\n''memote report snapshot --json --filename 05-automatic_curations/memote_final/C_beijerinckii_13821.html 05-automatic_curations/03-annotations/models/C_beijerinckii_13821.xml &> /dev/null'
bash -c ''$'\n''echo '\''Creating report for C_beijerinckii_791'\'''$'\n''memote report snapshot --json --filename 05-automatic_curations/memote_final/C_beijerinckii_791.html 05-automatic_curations/03-annotations/models/C_beijerinckii_791.xml &> /dev/null'
Creating report for C_beijerinckii_13821
Creating report for C_beijerinckii_791
